In [2]:
import pandas as pd
import numpy as np

# 1. Load data
df = pd.read_csv("/kaggle/input/datasets/anantthakkur/diabetic-data/diabetic_data (1).csv")
raw_count = len(df)

# 2. Handle Test Case: Missing Patient ID & Duplicates
df = df.dropna(subset=["patient_nbr", "encounter_id"])
df = df.drop_duplicates(subset=["encounter_id"])

# 3. Handle Test Case: Transfers / Expired patients
# In UCI Diabetes dataset, discharge_disposition_id 11, 19, 20, 21 = Expired; 2, 3, 4, 5 = Transfers
excluded_dispositions = [11, 19, 20, 21]
df["is_transfer"] = df["discharge_disposition_id"].isin([2, 3, 4, 5]).astype(int)
df = df[~df["discharge_disposition_id"].isin(excluded_dispositions)].copy()

# 4. Standardize Diagnosis Group (Primary Diagnosis)
def map_diag(code):
    code = str(code)
    if code.startswith("250"): return "Diabetes"
    elif code.startswith(("39", "40", "41", "42", "43", "44", "45")): return "Circulatory"
    elif code.startswith(("46", "47", "48", "49", "50", "51")): return "Respiratory"
    elif code.startswith(("52", "53", "54", "55", "56", "57")): return "Digestive"
    else: return "Other"

df["diagnosis_group"] = df["diag_1"].apply(map_diag)

# 5. Synthesize Admission & Discharge Dates (to enable SQL 30-day self-join)
df = df.sort_values(["patient_nbr", "encounter_id"]).reset_index(drop=True)
base_date = pd.Timestamp("2024-01-01")
np.random.seed(42)

# Assign sequential dates per patient
adm_dates = []
prev_patient = None
curr_date = base_date

for _, row in df.iterrows():
    if row["patient_nbr"] != prev_patient:
        curr_date = base_date + pd.Timedelta(days=int(np.random.randint(0, 180)))
    else:
        gap = int(np.random.randint(5, 28)) if row["readmitted"] == "<30" else int(np.random.randint(35, 120))
        curr_date = curr_date + pd.Timedelta(days=gap)
    adm_dates.append(curr_date)
    curr_date = curr_date + pd.Timedelta(days=int(row["time_in_hospital"]))
    prev_patient = row["patient_nbr"]

df["admission_date"] = adm_dates
df["discharge_date"] = df["admission_date"] + pd.to_timedelta(df["time_in_hospital"], unit="D")

# 6. Python 30-Day Readmission Flag Logic (Handles Multiple Admissions)
df = df.sort_values(["patient_nbr", "admission_date"])
df["next_admission_date"] = df.groupby("patient_nbr")["admission_date"].shift(-1)
df["days_to_readmit"] = (df["next_admission_date"] - df["discharge_date"]).dt.days
df["readmit_30d_flag"] = ((df["days_to_readmit"] >= 1) & (df["days_to_readmit"] <= 30) & (df["is_transfer"] == 0)).astype(int)

# 7. Export Clean Data & 20-Patient Verification Sample
cols = ["encounter_id", "patient_nbr", "age", "gender", "diagnosis_group", 
        "time_in_hospital", "is_transfer", "admission_date", "discharge_date", "readmit_30d_flag"]
df[cols].to_csv("clean_admissions.csv", index=False)

# Save 20 patients with multiple visits for manual verification
multi_patients = df[df.duplicated("patient_nbr", keep=False)]["patient_nbr"].unique()[:20]
df[df["patient_nbr"].isin(multi_patients)][cols + ["days_to_readmit"]].to_csv("verify_20_patients.csv", index=False)

print(f"Raw rows: {raw_count} | Clean rows: {len(df)} | 30-Day Readmission Rate: {df['readmit_30d_flag'].mean()*100:.2f}%")

Raw rows: 101766 | Clean rows: 100114 | 30-Day Readmission Rate: 4.06%


In [3]:
import pandas as pd

# Load the cleaned file already in /kaggle/working
df = pd.read_csv("/kaggle/working/clean_admissions.csv")

total_clean = len(df)
transfers = int(df["is_transfer"].sum())
readmits = int(df["readmit_30d_flag"].sum())
rate = (readmits / total_clean) * 100

print("=== DATA QUALITY & READMISSION SUMMARY ===")
print(f"Clean Eligible Encounters : {total_clean:,}")
print(f"Transfer Encounters (Flag): {transfers:,}")
print(f"30-Day Readmissions       : {readmits:,}")
print(f"30-Day Readmission Rate   : {rate:.2f}%\n")

print("=== READMISSION RATE BY DIAGNOSIS GROUP ===")
diag_summary = df.groupby("diagnosis_group").agg(
    total_admissions=("encounter_id", "count"),
    readmissions_30d=("readmit_30d_flag", "sum")
)
diag_summary["readmission_rate_pct"] = (diag_summary["readmissions_30d"] * 100 / diag_summary["total_admissions"]).round(2)
print(diag_summary.sort_values("readmission_rate_pct", ascending=False), "\n")

print("=== READMISSION RATE BY AGE GROUP ===")
age_summary = df.groupby("age").agg(
    total_admissions=("encounter_id", "count"),
    readmissions_30d=("readmit_30d_flag", "sum")
)
age_summary["readmission_rate_pct"] = (age_summary["readmissions_30d"] * 100 / age_summary["total_admissions"]).round(2)
print(age_summary.sort_values("readmission_rate_pct", ascending=False))

=== DATA QUALITY & READMISSION SUMMARY ===
Clean Eligible Encounters : 100,114
Transfer Encounters (Flag): 18,081
30-Day Readmissions       : 4,067
30-Day Readmission Rate   : 4.06%

=== READMISSION RATE BY DIAGNOSIS GROUP ===
                 total_admissions  readmissions_30d  readmission_rate_pct
diagnosis_group                                                          
Diabetes                     8693               547                  6.29
Digestive                    9183               426                  4.64
Respiratory                 10091               457                  4.53
Circulatory                 29834              1148                  3.85
Other                       42313              1489                  3.52 

=== READMISSION RATE BY AGE GROUP ===
          total_admissions  readmissions_30d  readmission_rate_pct
age                                                               
[20-30)               1650               152                  9.21
[30-40)       